# Raw data and cohort definition


It writes metadata/samples.csv, which the papermill driver reads to select
the samples passed to notebook 001.

## Setup

In [1]:
import pandas as pd
from cap.paths import (display_path, raw_data_dir, figures_dir,
                       dataframe_dir, interim_head_dir, sample_sheet, features_file)
from cap.metadata import validate_sample_sheet
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 80)

raw_dir  = raw_data_dir()
fig_dir  = figures_dir()
df_dir   = dataframe_dir()
head_dir = interim_head_dir()
features_file = features_file()
sample_sheet = sample_sheet()
print("raw counts:", display_path(raw_dir))

raw counts: data/raw/raw_counts


In [2]:
assert raw_dir.is_dir(), f"raw counts not found at {raw_dir}; set CAP_RAW_DATA"
assert features_file.is_file(), f"features.tsv not found at {features_file}"

In [3]:
def sample_paths(sample_id):
    return (raw_dir / f"{sample_id}-matrix.mtx.gz",
            raw_dir / f"{sample_id}-barcodes.tsv.gz")

In [4]:
matrices = sorted(raw_dir.glob("*-matrix.mtx.gz"))
barcodes = sorted(raw_dir.glob("*-barcodes.tsv.gz"))

assert matrices, f"no matrix files under {display_path(raw_dir)}"
assert len(matrices) == len(barcodes), (len(matrices), len(barcodes))

print(f"raw counts directory : {display_path(raw_dir)}")
print(f"matrix files         : {len(matrices)}")
print(f"barcode files        : {len(barcodes)}")
print(f"shared features.tsv  : {display_path(features_file)}")

raw counts directory : data/raw/raw_counts
matrix files         : 69
barcode files        : 69
shared features.tsv  : data/raw/features.tsv


In [5]:
accessions = sorted({p.name.split("_")[0][:7] for p in matrices})
print("accession prefixes present:", accessions)

example = matrices[0].name
print(f"\nexample filename: {example}")

accession prefixes present: ['GSM4909']

example filename: GSM4909253_N-PM0092-Total-matrix.mtx.gz


In [6]:
CONDITION_LABELS = {"N": "N", "TN": "TN", "HER2": "HER2+", "ER": "ER+"}
def parse_sample_name(stem):
    """Turn 'GSM4909285_TN-B1-MH4031' into structured metadata."""
    conditions = set(CONDITION_LABELS) | {"mER", "B1"}   # raw codes as they appear in the name
    sort_labels = {"Total", "Epi"}
    site_labels = {"LN"}

    gsm, _, rest = stem.partition("_")
    tokens = rest.split("-")

    condition = tokens[0]
    if condition not in conditions:
        raise ValueError(f"unrecognised condition {condition!r} in {stem!r}")

    # 'TN-B1-MH4031' is a BRCA1-mutant tumour; 'B1-KCF0894' is preneoplastic tissue, where B1 is itself the condition.
    idx, brca1 = 1, condition == "B1"
    if len(tokens) > 1 and tokens[1] == "B1":
        brca1, idx = True, 2

    patient = tokens[idx]
    qualifiers = tokens[idx + 1:]

    return {
        "sample_id": stem,
        "gsm": gsm,
        "condition": CONDITION_LABELS.get(condition, condition),
        "patient": patient,
        "brca1_carrier": brca1,
        "sorting": next((q for q in qualifiers if q in sort_labels), "unsorted"),
        "site": "lymph_node" if any(q in site_labels for q in qualifiers) else "primary",
        "replicate": next((q for q in qualifiers
                           if q not in sort_labels and q not in site_labels), ""),
    }

In [7]:
samples = pd.DataFrame(
    [parse_sample_name(p.name.replace("-matrix.mtx.gz", "")) for p in matrices]
)

print(f"{len(samples)} samples\n")
print(pd.crosstab(samples["condition"], samples["sorting"]).to_string())
print()
print(pd.crosstab(samples["condition"], samples["site"]).to_string())

69 samples

sorting    Epi  Total  unsorted
condition                      
B1           0      0         4
ER+          0      0        24
HER2+        0      0         6
N           11     13         0
TN           0      0         8
mER          0      0         3

site       lymph_node  primary
condition                     
B1                  0        4
ER+                 6       18
HER2+               0        6
N                   0       24
TN                  0        8
mER                 1        2


In [8]:
def decide_inclusion(row, has_total):
    """rules are applied in order, so a sample is counted under the first that matches"""
    if row["condition"] == "mER":
        return False, "male breast cancer; distinct biology and sex chromosome complement"

    if row["condition"] == "B1":
        return False, "BRCA1 preneoplastic tissue; neither normal nor tumour"

    if row["site"] == "lymph_node":
        return False, "lymph node metastasis; cohort restricted to primary tissue"

    if row["sorting"] == "Epi":
        if row["patient"] in has_total:
            return False, "epithelially sorted; unsorted -Total sample used for this patient"
        return False, ("epithelially sorted; cell-type composition not comparable "
                       "with the unsorted -Total samples in the rest of the cohort")

    return True, ""

In [9]:
has_total = set(samples.loc[samples["sorting"] == "Total", "patient"])

samples["include"], samples["exclusion_reason"] = zip(
    *samples.apply(lambda row: decide_inclusion(row, has_total), axis=1)
)

included = samples[samples["include"]]

print(f"{len(samples)} samples found -> {len(included)} included\n")
print("included:")
print(included.groupby("condition").agg(
    n_samples=("sample_id", "size"),
    n_patients=("patient", "nunique"),
    n_brca1=("brca1_carrier", "sum"),
).to_string())

print("\nexcluded:")
for reason, n in samples.loc[~samples["include"], "exclusion_reason"].value_counts().items():
    print(f"  {n:3d}  {reason}")

69 samples found -> 45 included

included:
           n_samples  n_patients  n_brca1
condition                                
ER+               18          17        0
HER2+              6           6        0
N                 13          13        0
TN                 8           8        4

excluded:
    6  epithelially sorted; unsorted -Total sample used for this patient
    6  lymph node metastasis; cohort restricted to primary tissue
    5  epithelially sorted; cell-type composition not comparable with the unsorted -Total samples in the rest of the cohort
    4  BRCA1 preneoplastic tissue; neither normal nor tumour
    3  male breast cancer; distinct biology and sex chromosome complement


In [10]:
repeated = (included[included.duplicated("patient", keep=False)]
            .sort_values(["patient", "sample_id"]))

print(f"{len(included)} samples from {included['patient'].nunique()} distinct patients\n")

for patient, group in repeated.groupby("patient"):
    conditions = sorted(set(group["condition"]))
    flag = "    -> spans conditions" if len(conditions) > 1 else ""
    print(f"  {patient:10s} {list(group['sample_id'])}{flag}")

45 samples from 42 distinct patients

  MH0029     ['GSM4909305_ER-MH0029-7C', 'GSM4909306_ER-MH0029-9C']
  MH0064     ['GSM4909263_N-MH0064-Total', 'GSM4909313_ER-MH0064-T']    -> spans conditions
  MH0114     ['GSM4909284_TN-MH0114-T2', 'GSM4909299_ER-MH0114-T3']    -> spans conditions


##  Write the sample sheet

In [11]:
# both files must exist for everything we intend to load
def files_present(sample_id):
    matrix, barcode = sample_paths(sample_id)
    return matrix.exists() and barcode.exists()

samples["files_present"] = samples["sample_id"].map(files_present)
missing = samples.loc[samples["include"] & ~samples["files_present"], "sample_id"]
assert missing.empty, f"missing raw files for: {list(missing)}"

samples = samples.sort_values(["condition", "patient", "sample_id"]).reset_index(drop=True)
sample_sheet.parent.mkdir(parents=True, exist_ok=True)
validate_sample_sheet(samples)   # samples still has sample_id as a column here
samples.to_csv(sample_sheet, index=False)

# the cohort reported in the methods; these fail loudly if the rules or the download ever change
assert len(samples) == 69, len(samples)
assert samples["include"].sum() == 45, samples["include"].sum()
assert included["patient"].nunique() == 42, included["patient"].nunique()

print(f"written: {display_path(sample_sheet)}")
print(f"  {len(samples)} rows, {samples['include'].sum()} included")
print(f"  columns: {', '.join(samples.columns)}")

written: Metadata/samples.csv
  69 rows, 45 included
  columns: sample_id, gsm, condition, patient, brca1_carrier, sorting, site, replicate, include, exclusion_reason, files_present


In [13]:
import session_info


In [14]:
session_info.show()